In [1]:
# Install (agar pehle nahi kiya)
!pip install gensim

# Imports
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 734.4 kB/s eta 0:00:33
    --------------------------------------- 0.5/24.4 MB 734.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/24.4 MB 675.5 kB/s eta 0:00:35
   - -------------------------------------- 0.8/24.4 MB 675.5 kB/s eta 0:00:35
   - -------------------------------------- 1.0/24.4 MB 678.3 kB/s eta 0:00:35
   -- ------------------------------------- 1.3/24.4 MB 749.5 kB/s eta 0:00:31
   -- ------------------------------------- 1.6/24.4 MB 783.5 kB/s eta 0:00:30
   -- ------------------------------------- 1.6/24.4 MB 783.5 kB/s eta 0:00:30
   --- ---

In [2]:
data = [
    # Eligibility
    ("am i eligible for credit card", "eligibility"),
    ("can i get approved for loan", "eligibility"),
    ("will my application pass", "eligibility"),
    ("can i get a credit card", "eligibility"),
    ("is my profile eligible", "eligibility"),

    # Risk
    ("what is my risk", "risk"),
    ("am i a risky customer", "risk"),
    ("what are chances of default", "risk"),
    ("how risky am i", "risk"),
    ("tell my risk score", "risk"),

    # Improvement
    ("how to improve credit score", "improvement"),
    ("tips to increase score", "improvement"),
    ("how can i reduce risk", "improvement"),
    ("ways to improve my credit", "improvement"),
    ("how to get better score", "improvement"),
]

In [3]:
print(len(data))
print(data[:3])

15
[('am i eligible for credit card', 'eligibility'), ('can i get approved for loan', 'eligibility'), ('will my application pass', 'eligibility')]


In [4]:
def tokenize(text):
    return text.lower().split()

# convert sentences to tokens
sentences = [tokenize(t[0]) for t in data]

# check output
print(sentences[:3])

[['am', 'i', 'eligible', 'for', 'credit', 'card'], ['can', 'i', 'get', 'approved', 'for', 'loan'], ['will', 'my', 'application', 'pass']]


In [5]:
from gensim.models import Word2Vec

# Train model
w2v_model = Word2Vec(
    sentences,
    vector_size=100,   # vector size (important 🔥)
    window=5,          # context words
    min_count=1        # small dataset ke liye 1
)

# Check vector of a word
print(w2v_model.wv['credit'][:10])   # first 10 values

[-0.00872748  0.00213016 -0.00087354 -0.00931909 -0.00942814 -0.00141072
  0.00443241  0.00370407 -0.00649869 -0.00687307]


In [6]:
print(w2v_model.wv.most_similar('credit'))

[('risky', 0.1667264699935913), ('pass', 0.16260264813899994), ('how', 0.1388544738292694), ('score', 0.1314060389995575), ('reduce', 0.0974535346031189), ('are', 0.09636400640010834), ('eligible', 0.0717582106590271), ('to', 0.06404781341552734), ('can', 0.060420017689466476), ('for', 0.04762953892350197)]


In [7]:
import numpy as np

def sentence_vector(sentence):
    words = tokenize(sentence)
    vectors = []

    for w in words:
        if w in w2v_model.wv:
            vectors.append(w2v_model.wv[w])

    # agar koi word match na kare
    if len(vectors) == 0:
        return np.zeros(100)

    # average of all word vectors
    return np.mean(vectors, axis=0)

In [8]:
vec = sentence_vector("am i eligible for credit card")

print(vec.shape)
print(vec[:10])   # first 10 values

(100,)
[-0.00308438 -0.00046925  0.00102455  0.00132106 -0.00192583 -0.00402577
  0.00108621  0.00530743 -0.00355678 -0.00175044]


In [9]:
from sklearn.linear_model import LogisticRegression

# Input (X) = sentence vectors
X = np.array([sentence_vector(t[0]) for t in data])

# Output (y) = labels
y = [t[1] for t in data]

# Train model
intent_model = LogisticRegression()
intent_model.fit(X, y)

print("Model trained successfully ✅")

Model trained successfully ✅


In [10]:
test = "can i get a loan"

vec = sentence_vector(test).reshape(1, -1)
pred = intent_model.predict(vec)

print("Intent:", pred[0])

Intent: eligibility


In [11]:
def chatbot(user_input, user_data=None):
    # intent predict karo
    vec = sentence_vector(user_input).reshape(1, -1)
    intent = intent_model.predict(vec)[0]

    # response logic
    if intent == "eligibility":
        if user_data is not None:
            prob = model.predict_proba(user_data)[0][1]
            return f"Approval chance: {round((1-prob)*100,2)}%"
        return "Eligibility depends on your income, credit score, and history."

    elif intent == "risk":
        if user_data is not None:
            prob = model.predict_proba(user_data)[0][1]
            return f"Risk of default: {round(prob*100,2)}%"
        return "Risk depends on your financial behavior."

    elif intent == "improvement":
        return "Pay bills on time, reduce credit usage, and avoid multiple loans."

    else:
        return "Sorry, I didn't understand."

In [12]:
print(chatbot("am i eligible for credit card"))
print(chatbot("what is my risk"))
print(chatbot("how to improve my score"))

Eligibility depends on your income, credit score, and history.
Risk depends on your financial behavior.
Pay bills on time, reduce credit usage, and avoid multiple loans.
